# 양자화 행렬연산 체험 실습 (초보자용)

이 실습의 목표는 **양자화가 실제로 행렬연산을 어떻게 바꾸는지** 직접 체험하는 것입니다.

오늘 해볼 것:

1. `float` 행렬곱 해보기  
2. 가중치만 `INT8`로 양자화해 보기  
3. 입력과 가중치를 둘 다 양자화해서 **정수 행렬곱** 해보기  
4. `INT8`과 `INT4`의 오차 차이 보기  
5. outlier(유난히 큰 값)가 있으면 왜 양자화가 어려워지는지 보기

---

## 핵심 아이디어

원래 모델 계산은 대체로 이런 식입니다.

\[
Y = XW
\]

양자화하면 대충 이런 식으로 바뀝니다.

\[
X \approx s_x X_q, \quad W \approx s_w W_q
\]

그래서 계산은

\[
Y \approx (s_x s_w)(X_q W_q)
\]

즉,

- 원래는 **실수 행렬곱**
- 양자화 후에는 **정수 행렬곱 + scale 복원**

으로 이해하면 됩니다.

In [1]:

# 실습에 필요한 라이브러리만 불러옵니다.
import numpy as np

# 출력 결과를 보기 좋게 하기 위한 설정입니다.
np.set_printoptions(precision=4, suppress=True)

# 같은 결과가 나오도록 난수를 고정합니다.
np.random.seed(42)

## 1. 실수(float) 행렬곱 먼저 해보기

양자화를 하기 전에 먼저 **원래 계산**이 어떻게 생겼는지 확인해 봅시다.

- `X`: 입력 행렬
- `W`: 가중치 행렬
- `Y = X @ W`: 출력 행렬

In [2]:

# 입력 행렬 X: (2, 4)
# -> 배치가 2개이고, 각 입력 벡터의 길이가 4라고 생각하면 됩니다.
X = np.random.randn(2, 4).astype(np.float32)

# 가중치 행렬 W: (4, 3)
# -> 입력 4차원을 받아서 출력 3차원으로 바꾸는 선형층(weight)처럼 생각하면 됩니다.
W = np.random.randn(4, 3).astype(np.float32)

# 원래 float 기준의 행렬곱 결과
Y_float = X @ W

print("입력 X")
print(X)

print("\n가중치 W")
print(W)

print("\n원래 float 행렬곱 결과 Y_float = X @ W")
print(Y_float)

입력 X
[[ 0.4967 -0.1383  0.6477  1.523 ]
 [-0.2342 -0.2341  1.5792  0.7674]]

가중치 W
[[-0.4695  0.5426 -0.4634]
 [-0.4657  0.242  -1.9133]
 [-1.7249 -0.5623 -1.0128]
 [ 0.3142 -0.908  -1.4123]]

원래 float 행렬곱 결과 Y_float = X @ W
[[-0.8074 -1.5111 -2.7726]
 [-2.2639 -1.7685 -2.1268]]


## 2. 실수 행렬을 INT8로 양자화해 보기

양자화의 핵심은 **실수값을 정수 격자(grid)에 맞춰 저장**하는 것입니다.

여기서는 가장 단순한 방식인 **대칭 양자화(symmetric quantization)** 를 사용합니다.

### 방식
1. 행렬의 절댓값 최댓값을 찾는다.
2. 그 값을 기준으로 `scale`을 만든다.
3. `실수 / scale` 해서 정수 범위로 보낸다.
4. 반올림한다.
5. `int8` 범위 `[-127, 127]`로 자른다.
6. scale 값은 보통 max(abs(실수)) / 127 으로 구함

int8의 실제 저장 범위는 [-128, 127]이지만,
대칭 양자화에서는 0을 기준으로 대칭을 맞추기 위해
[-127, 127]을 사용합니다. (-128은 대칭 짝이 없어 제외)

복원할 때는 다시 `정수 * scale`을 해주면 됩니다.

In [3]:

def quantize_symmetric(x, num_bits=8):
    '''
    대칭 양자화 함수

    x        : 원본 float numpy 배열
    num_bits : 몇 비트로 양자화할지 (예: 8, 4)

    반환값
    ------
    x_q      : 양자화된 정수 배열(int32로 저장)
    scale    : 복원할 때 사용할 스케일 값
    qmin/qmax: 양자화 가능한 최소/최대 정수 범위
    '''
    # 예: 8비트면 -127 ~ 127, 4비트면 -7 ~ 7 사용
    qmax = (2 ** (num_bits - 1)) - 1
    qmin = -qmax

    # 모든 값이 0일 때 scale이 0이 되지 않도록 예외처리
    max_abs = np.max(np.abs(x))
    scale = max_abs / qmax if max_abs != 0 else 1.0

    # 실수를 scale로 나눠 정수 범위로 옮긴 뒤 반올림
    x_q = np.round(x / scale)

    # 정수 범위를 넘지 않도록 잘라줍니다.
    x_q = np.clip(x_q, qmin, qmax).astype(np.int32)
    return x_q, scale, qmin, qmax


def dequantize(x_q, scale):
    '''
    양자화된 정수 배열을 다시 float로 복원하는 함수
    '''
    return x_q.astype(np.float32) * scale

In [4]:

# 가중치 W를 INT8로 양자화해 봅니다.
W_q8, scale_w8, qmin8, qmax8 = quantize_symmetric(W, num_bits=8)

# 다시 float로 복원해 봅니다.
W_deq8 = dequantize(W_q8, scale_w8)

print("INT8 범위:", qmin8, "~", qmax8)
print("scale_w8:", scale_w8)

print("\n원본 W")
print(W)

print("\n양자화된 W_q8")
print(W_q8)

print("\n복원된 W_deq8")
print(W_deq8)

# 원본과 복원본의 차이를 평균 절대 오차(MAE)로 봅니다.
mae_W_8 = np.mean(np.abs(W - W_deq8))
print("\nW의 복원 오차(MAE):", mae_W_8)

INT8 범위: -127 ~ 127
scale_w8: 0.015065199

원본 W
[[-0.4695  0.5426 -0.4634]
 [-0.4657  0.242  -1.9133]
 [-1.7249 -0.5623 -1.0128]
 [ 0.3142 -0.908  -1.4123]]

양자화된 W_q8
[[ -31   36  -31]
 [ -31   16 -127]
 [-114  -37  -67]
 [  21  -60  -94]]

복원된 W_deq8
[[-0.467   0.5423 -0.467 ]
 [-0.467   0.241  -1.9133]
 [-1.7174 -0.5574 -1.0094]
 [ 0.3164 -0.9039 -1.4161]]

W의 복원 오차(MAE): 0.0028635226


## 3. 가중치만 INT8 양자화해서 결과 비교하기

실제 추론에서는 **원래 계산 결과와 양자화 후 계산 결과가 얼마나 다른지**가 중요합니다.

여기서는

- 원래: `Y_float = X @ W`
- 양자화 후: `Y_w8 = X @ W_deq8`

를 비교해 보겠습니다.

즉, **가중치만 압축해서 써도 결과가 꽤 비슷한가?**를 보는 단계입니다.

In [5]:

# 가중치만 INT8 양자화 후 복원해서 사용
Y_w8 = X @ W_deq8

print("원래 float 결과 Y_float")
print(Y_float)

print("\n가중치만 INT8 양자화 후 결과 Y_w8")
print(Y_w8)

mae_y_w8 = np.mean(np.abs(Y_float - Y_w8))
print("\n출력 결과 오차(MAE):", mae_y_w8)

원래 float 결과 Y_float
[[-0.8074 -1.5111 -2.7726]
 [-2.2639 -1.7685 -2.1268]]

가중치만 INT8 양자화 후 결과 Y_w8
[[-0.7979 -1.5017 -2.778 ]
 [-2.2507 -1.7574 -2.1235]]

출력 결과 오차(MAE): 0.008660823


## 4. 입력과 가중치를 둘 다 양자화해서 정수 행렬곱 해보기

이 단계가 가장 중요합니다.

양자화의 진짜 핵심은 아래입니다.

- 입력 `X`를 정수로 바꾸고
- 가중치 `W`도 정수로 바꾸고
- **정수끼리 행렬곱** 한 뒤
- 마지막에 `scale_x * scale_w`를 곱해 복원

즉,  
**실수 행렬곱 → 정수 행렬곱 + 스케일 복원**으로 바뀌는 것을 직접 보는 실습입니다.

In [6]:

# 입력 X도 INT8로 양자화합니다.
X_q8, scale_x8, _, _ = quantize_symmetric(X, num_bits=8)

# 정수 행렬곱
# 실제 하드웨어에서는 int8 x int8 후 int32 누산(accumulation)하는 식으로 동작합니다.
# 여기서는 그 느낌을 살리기 위해 int32로 바꿔서 곱합니다.

# int8 × int8 = 최대 127×127 = 16129,
# 이를 N번 더하면 int8 범위(-128~127)를 쉽게 초과(overflow)합니다.
# 실제 하드웨어도 int32로 누산(accumulation)하므로 여기서도 int32로 캐스팅합니다.
Y_int8 = X_q8.astype(np.int32) @ W_q8.astype(np.int32)

# 다시 float로 복원
Y_both8 = Y_int8.astype(np.float32) * (scale_x8 * scale_w8)

print("양자화된 입력 X_q8")
print(X_q8)

print("\n양자화된 가중치 W_q8")
print(W_q8)

print("\n정수 행렬곱 결과 Y_int8")
print(Y_int8)

print("\n복원된 결과 Y_both8")
print(Y_both8)

mae_y_both8 = np.mean(np.abs(Y_float - Y_both8))
print("\n원래 float 결과와의 오차(MAE):", mae_y_both8)

양자화된 입력 X_q8
[[ 40 -11  52 122]
 [-19 -19 127  62]]

양자화된 가중치 W_q8
[[ -31   36  -31]
 [ -31   16 -127]
 [-114  -37  -67]
 [  21  -60  -94]]

정수 행렬곱 결과 Y_int8
[[ -4265  -7980 -14795]
 [-11998  -9407 -11335]]

복원된 결과 Y_both8
[[-0.799  -1.4949 -2.7716]
 [-2.2476 -1.7622 -2.1234]]

원래 float 결과와의 오차(MAE): 0.008609335


## 5. INT8과 INT4 비교하기

이번에는 비트 수를 더 줄여서 `INT4`도 비교해 보겠습니다.

비트 수가 줄어들면:

- 저장 공간은 더 줄어들 수 있지만
- 표현 가능한 정수 칸 수가 적어져서
- 오차가 더 커지는 경우가 많습니다.

보통:
- INT8: 꽤 안정적
- INT4: 더 공격적인 압축

In [7]:

# INT4 양자화
X_q4, scale_x4, qmin4, qmax4 = quantize_symmetric(X, num_bits=4)
W_q4, scale_w4, _, _ = quantize_symmetric(W, num_bits=4)

# 정수 행렬곱 후 복원
Y_int4 = X_q4.astype(np.int32) @ W_q4.astype(np.int32)
Y_both4 = Y_int4.astype(np.float32) * (scale_x4 * scale_w4)

# 오차 계산
mae_y_both4 = np.mean(np.abs(Y_float - Y_both4))

print("INT4 범위:", qmin4, "~", qmax4)

print("\nINT8 양자화 결과 오차(MAE):", mae_y_both8)
print("INT4 양자화 결과 오차(MAE):", mae_y_both4)

print("\n원래 float 결과")
print(Y_float)

print("\nINT4 복원 결과")
print(Y_both4)

INT4 범위: -7 ~ 7

INT8 양자화 결과 오차(MAE): 0.008609335
INT4 양자화 결과 오차(MAE): 0.066284746

원래 float 결과
[[-0.8074 -1.5111 -2.7726]
 [-2.2639 -1.7685 -2.1268]]

INT4 복원 결과
[[-0.8016 -1.4799 -2.7132]
 [-2.1582 -1.6032 -2.0965]]


### 생각해 보기
왜 INT4가 INT8보다 대체로 오차가 더 클까요?

힌트:
- INT8은 가능한 정수 칸이 더 많습니다.
- INT4는 가능한 정수 칸이 훨씬 적습니다.
- 즉, 실수값을 더 거칠게 반올림하게 됩니다.

이 코드에서 사용하는 대칭 양자화 기준:
- INT8: -127 ~ 127 → 총 255칸
- INT4: -7  ~ 7   → 총 15칸

## 6. outlier가 있으면 왜 양자화가 어려워질까?

양자화에서 자주 등장하는 문제 중 하나가 **outlier(유난히 큰 값)** 입니다.

예를 들어 행렬 대부분은 작은 값인데,  
딱 하나만 엄청 큰 값이 있으면 어떻게 될까요?

- 그 큰 값 때문에 `scale`이 커집니다.
- 그러면 작은 값들은 상대적으로 더 거칠게 반올림됩니다.
- 결과적으로 전체 오차가 커질 수 있습니다.

In [8]:

# 원본 W를 복사해서 outlier를 하나 넣어봅니다.
W_outlier = W.copy()
W_outlier[0, 0] = 20.0  # 유난히 큰 값 하나 추가

# 원래 float 계산
Y_float_outlier = X @ W_outlier

# INT8 양자화
W_out_q8, scale_out8, _, _ = quantize_symmetric(W_outlier, num_bits=8)
W_out_deq8 = dequantize(W_out_q8, scale_out8)

# 가중치만 양자화해서 계산
Y_out_q = X @ W_out_deq8

mae_outlier = np.mean(np.abs(Y_float_outlier - Y_out_q))

print("outlier가 들어간 W_outlier")
print(W_outlier)

print("\noutlier가 들어가면서 커진 scale")
print(scale_out8)

print("\n양자화된 W_out_q8")
print(W_out_q8)

print("\n원래 결과 Y_float_outlier")
print(Y_float_outlier)

print("\n양자화 후 결과 Y_out_q")
print(Y_out_q)

print("\noutlier가 있을 때 출력 오차(MAE):", mae_outlier)

outlier가 들어간 W_outlier
[[20.      0.5426 -0.4634]
 [-0.4657  0.242  -1.9133]
 [-1.7249 -0.5623 -1.0128]
 [ 0.3142 -0.908  -1.4123]]

outlier가 들어가면서 커진 scale
0.15748031

양자화된 W_out_q8
[[127   3  -3]
 [ -3   2 -12]
 [-11  -4  -6]
 [  2  -6  -9]]

원래 결과 Y_float_outlier
[[ 9.3601 -1.5111 -2.7726]
 [-7.0569 -1.7685 -2.1268]]

양자화 후 결과 Y_out_q
[[ 9.3573 -1.656  -2.744 ]
 [-7.0664 -1.9043 -2.0268]]

outlier가 있을 때 출력 오차(MAE): 0.07026569


## 7. 직접 바꿔보는 연습

아래를 직접 바꿔보면서 결과를 비교해 보세요.

### 연습 1
`W_outlier[0, 0] = 20.0` 을

- `5.0`
- `50.0`
- `100.0`

으로 바꿔 보세요.

### 연습 2
`num_bits=8` 을 `num_bits=6`, `num_bits=4`로 바꿔 보세요.

### 연습 3
행렬 크기를 더 크게 바꿔 보세요.

예:
- `X`: `(4, 8)`
- `W`: `(8, 5)`

### 연습 4
가중치만 양자화했을 때와  
입력+가중치를 모두 양자화했을 때의 오차를 비교해 보세요.

## 8. 한 줄 정리

이 실습에서 기억할 핵심은 이것입니다.

- 양자화는 단순히 저장 비트 수(메모리)를 줄이는 작업이 아닙니다.
- 실수 행렬연산을 정수 행렬연산으로 바꾸는 과정이며, 그 과정에서 발생하는 오차를 얼마나 줄이느냐가 핵심입니다.
- 그리고 마지막에 `scale`을 곱해 원래 값에 가깝게 복원합니다.
- 비트 수가 적을수록 더 작아지지만, 오차는 커질 가능성이 높습니다.
- outlier가 있으면 scale이 커져서 양자화가 더 어려워질 수 있습니다.